# **Fine-Tuning a Sentiment Analysis LLM - Llama 2**
**Using a specific dataset with QLORA strategy**

# 1 - Installing required packages

In [1]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.9 MB/s eta 0:00:00


In [2]:
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 115.0 MB/s eta 0:00:00


In [3]:
!pip install -q accelerate==1.9.0 peft==0.16.0 transformers==4.54.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 146.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
!pip install -q trl==0.20.0 gradio==5.38.2 protobuf scipy==1.16.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires grad

In [5]:
!pip install -q sentencepiece==0.2.0 tokenizers==0.21.2 datasets==4.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.11.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is

In [6]:
# Imports
import os
import torch
import datasets
import pandas as pd
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser,
                          TrainingArguments, pipeline, logging)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings("ignore")

# 2 - Add configuration to ease development

In [7]:
# Defining logging verbosity
logging.set_verbosity(logging.CRITICAL)

In [8]:
# Checking GPU model available
if torch.cuda.is_available():
  print("GPUs: ", torch.cuda.device_count())
  print("GPU Name: ", torch.cuda.get_device_name(0))
  print("GPU Total Memory: ", torch.cuda.get_device_properties(0).total_memory/1e9)

GPUs:  1
GPU Name:  NVIDIA A100-SXM4-80GB
GPU Total Memory:  85.094825984


In [9]:
# Adding bckp block in order to reset GPU memory, if necessary
from numba import cuda
device = cuda.get_current_device()
device.reset()

In [10]:
!ls drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/

BKP
dataset_with_general.csv
DEPLOY__FINE_TUNING_SENTIMENT_ANALYSIS_LLM_LLAMA2.ipynb
deploy_requirements.txt
FINE_TUNING_SENTIMENT_ANALYSIS_LLM_LLAMA2.ipynb
saida


# 3 - Loading dataset for Fine-Tuning

In [11]:
# Definig dataset var
dataset_name = 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/dataset_with_general.csv'

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Load dataset
## --> The 'load_dataset' is excelent because it will convert the data in a format that LLM can understand
dataset_loaded = load_dataset('csv', data_files = dataset_name, delimiter = ',')

Generating train split: 0 examples [00:00, ? examples/s]

In [14]:
# Show dataset
print(dataset_loaded)

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 23057
    })
})


Let's inspect a few examples from your loaded dataset to understand its structure. This will help us determine if the data is formatted in a way that encourages the model to generate sentiment labels.

In [15]:
# Print the first 5 examples from the 'train' split of the dataset
for i in range(5):
    print(dataset_loaded['train'][i])

{'train': '<s>[INST] Very good documentary calling to question the morals of detention without trial and torture as a method to protect national security. A well made documentary. [/INST] Positive </s>'}
{'train': '<s>[INST] Yes! our product launch went perfectly. [/INST] Positive </s>'}
{'train': "<s>[INST] Cheaply made, unconvincing story of assorted characters pretending to be somebody else in wartime Poland. Supposed to be a comedy, but very few chuckles. Weak script does not help. It was Carole Lombard's last film, but she appeared in much more entertaining vehicles. Plus, in this one, her good looks had rather faded. [/INST] Negative </s>"}
{'train': "<s>[INST] Exceptional film Heartbreaking and warm.  You'd have to have your head buried in the sand for it not to effect you. The landscape is breathtaking and cinematography too.\n\nThat other humans wreck the planet with their greed and thoughtlessness should watch this. \n\nHardworking and so positive Hatidze is fabulous.\n\nWatc

# 4 - Defining params before Fine-Tuning

In [16]:
# Defining LORA params
lora_r = 32
lora_alpha = 16
lora_dropout = 0.1

In [17]:
# Defining QLORA params
use_4bit = True
bnb_4bit_compute_dtype = "bfloat16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

In [18]:
# Defining Fine-Tuning params
output_dir = "drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida"
num_train_epochs = 1
fp16 = False
bf16 = True
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "adamw_8bit"
lr_scheduler_type = "linear"
max_steps = -1
warmup_ratio = 0.03
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
seed = 3407
report_to = "none"
dataset_text_field = "train"
packing = False

In [19]:
# Defining extra fine-tuning params
## --> Related to batch padding, saving and logging
group_by_length = True
save_steps = 0
logging_steps = 400

# 5 - Fine-Tuning the LLM

5.1 - Training

In [20]:
# Defining extra important vars
repository_hf = "NousResearch/Llama-2-7b-chat-hf"
model_tuned = "drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned"

In [21]:
# Defining LORA config
peft_config = LoraConfig(lora_alpha = lora_alpha,
                         lora_dropout = lora_dropout,
                         r = lora_r,
                         bias = "none",
                         task_type = "CAUSAL_LM")

In [22]:
# Defining QLORA (quantization) config
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [23]:
# Load pre-trained model
model = AutoModelForCausalLM.from_pretrained(repository_hf,
                                              quantization_config = bnb_config,
                                              device_map = "auto")

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [24]:
# Dont need for cache or parallelism
model.config.use_cache = False
model.config.pretraining_tp = 1

In [25]:
# Loading tokenizer of pre-trained model
tokenizer = AutoTokenizer.from_pretrained(repository_hf, trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [26]:
# Defining traning params (1º version)
training_arguments = TrainingArguments(output_dir = output_dir,
                                       num_train_epochs = num_train_epochs,
                                       per_device_train_batch_size = per_device_train_batch_size,
                                       gradient_accumulation_steps = gradient_accumulation_steps,
                                       optim = optim,
                                       save_steps = save_steps,
                                       logging_steps = logging_steps,
                                       learning_rate = learning_rate,
                                       weight_decay = weight_decay,
                                       fp16 = fp16,
                                       bf16 = bf16,
                                       max_grad_norm = max_grad_norm,
                                       max_steps = max_steps,
                                       warmup_ratio = warmup_ratio,
                                       group_by_length = group_by_length,
                                       lr_scheduler_type = lr_scheduler_type
                      )

In [27]:
# Defining traning params (2º version)
sft_config = SFTConfig(per_device_train_batch_size = per_device_train_batch_size,
                       gradient_accumulation_steps = gradient_accumulation_steps,
                       warmup_steps = 5,
                       num_train_epochs = num_train_epochs,
                       learning_rate = learning_rate,
                       fp16 = fp16,
                       logging_steps = logging_steps,
                       optim = optim,
                       weight_decay = weight_decay,
                       lr_scheduler_type = lr_scheduler_type,
                       seed = seed,
                       output_dir = output_dir,
                       report_to = report_to,
                       dataset_text_field = dataset_text_field,
                       packing = packing
            )

In [28]:
# Creating the trainer object
model_trainer = SFTTrainer(
                            model = model,
                            train_dataset = dataset_loaded["train"],
                            peft_config = peft_config,
                            args = sft_config
                )

Adding EOS to train dataset:   0%|          | 0/23057 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/23057 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/23057 [00:00<?, ? examples/s]

In [29]:
# Training the fine-tuned model
%%time
model_trainer.train()

{'loss': 2.255, 'grad_norm': 0.5963860750198364, 'learning_rate': 0.00018631944444444444, 'num_tokens': 148786.0, 'mean_token_accuracy': 0.5247301058471203, 'epoch': 0.06938421509106678}
{'loss': 2.0877, 'grad_norm': 0.66828852891922, 'learning_rate': 0.00017243055555555555, 'num_tokens': 308177.0, 'mean_token_accuracy': 0.5452235224843025, 'epoch': 0.13876843018213356}
{'loss': 2.0539, 'grad_norm': 0.3065342903137207, 'learning_rate': 0.00015854166666666667, 'num_tokens': 457639.0, 'mean_token_accuracy': 0.5515744662284852, 'epoch': 0.20815264527320035}
{'loss': 2.0571, 'grad_norm': 0.5707964301109314, 'learning_rate': 0.00014465277777777778, 'num_tokens': 612189.0, 'mean_token_accuracy': 0.5509098102152348, 'epoch': 0.2775368603642671}
{'loss': 2.0197, 'grad_norm': 0.8017163276672363, 'learning_rate': 0.0001307638888888889, 'num_tokens': 763574.0, 'mean_token_accuracy': 0.5601387219876051, 'epoch': 0.3469210754553339}
{'loss': 2.0343, 'grad_norm': 0.4969397187232971, 'learning_rate':

TrainOutput(global_step=5765, training_loss=2.0198774087977016, metrics={'train_runtime': 2132.1775, 'train_samples_per_second': 10.814, 'train_steps_per_second': 2.704, 'total_flos': 1.8943206340664525e+17, 'train_loss': 2.0198774087977016})

In [30]:
# Saving tuned model
model_trainer.model.save_pretrained(model_tuned)

# 6 - Generating responses using the Fine-Tuned LLM

In [31]:
# Generating a prompt
prompt = "I feel like trying to cope with AI news an impossible task nowadays"

In [32]:
# Generating a pipe
pipe = pipeline(task = "text-generation",
                model = model,
                tokenizer = tokenizer)

In [33]:
# Executing the pipeline and extract result
# Update the prompt to explicitly ask for the sentiment
prompt_for_model = f"[INST] {prompt} [/INST]"
with torch.autocast("cuda", dtype=torch.bfloat16):
    result = pipe(prompt_for_model, max_new_tokens=5, do_sample=False, return_full_text=False)
print(result[0]["generated_text"])

 Negative


In [34]:
tests = [
    "This movie was a complete waste of time.",
    "Brilliant acting and a gripping story from start to finish.",
    "The plot was boring and the ending made no sense.",
    "One of the best films I have ever seen.",
    "I feel like trying to cope with AI news an impossible task nowadays.",
]

with torch.autocast("cuda", dtype=torch.bfloat16):
    for t in tests:
        out = pipe(f"[INST] {t} [/INST]", max_new_tokens=5, do_sample=False, return_full_text=False)
        print(f"{out[0]['generated_text'].strip():10} | {t}")

Negative   | This movie was a complete waste of time.
Positive   | Brilliant acting and a gripping story from start to finish.
Negative   | The plot was boring and the ending made no sense.
Positive   | One of the best films I have ever seen.
Negative   | I feel like trying to cope with AI news an impossible task nowadays.


# 7 - Saving Model for Deployment

In [35]:
# Saving the tokenizer (final)
tokenizer_final = AutoTokenizer.from_pretrained(repository_hf, trust_remote_code = True)
tokenizer_final.pad_token = tokenizer_final.eos_token
tokenizer_final.padding_side = "right"
tokenizer_final.save_pretrained(model_tuned+"/final/tokenizer_final_bf16")

('drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned/final/tokenizer_final_bf16/tokenizer_config.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned/final/tokenizer_final_bf16/special_tokens_map.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned/final/tokenizer_final_bf16/tokenizer.model',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned/final/tokenizer_final_bf16/added_tokens.json',
 'drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap07/saida/saida/model_tuned/final/tokenizer_final_bf16/tokenizer.json')

In [36]:
# Saving the model (final) - merge into a full-precision base
base = AutoModelForCausalLM.from_pretrained(repository_hf, torch_dtype=torch.bfloat16, device_map="auto")
model_tuned_final = PeftModel.from_pretrained(base, model_tuned).merge_and_unload()
model_tuned_final.save_pretrained(model_tuned + "/final/model_tuned_final_bf16")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]